In [1]:
import json
import re

In [2]:
from functions import *

In [3]:
def parse_ann_file(file_path):
    entities = {}
    relationships = []
    scope_or = []
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            entry_type = parts[0][0]
            if entry_type == 'T':  
                entity_id, entity_info = parts[0], parts[1:]
                label, position = entity_info[0].split(' ')[0], entity_info[0].split(' ')[1:]
                start, end = position[0], position[1]
                if ';' in start:
                    start = start.split(';')[0]
                if ';' in end:
                    end = end.split(';')[0]
                entities[entity_id] = {
                    'type': label,
                    'start': int(start),
                    'end': int(end),
                    'text': parts[2]
                }
            elif entry_type == 'R':  # Relationship
                relation_id, relation_info = parts[0], parts[1]
                relation_type, arg1, arg2 = relation_info.split(' ')[0], relation_info.split(' ')[1].split(':')[1], relation_info.split(' ')[2].split(':')[1]
                relationships.append({
                    'id': relation_id,
                    'type': relation_type,
                    'arg1': arg1,
                    'arg2': arg2
                })
            elif parts[0].startswith('*'):
                relationship_type = parts[1].split()[0]
                entity_ids = parts[1].split()[1:]
                scope_or.append({
                    'type': relationship_type,
                    'entities': entity_ids
                })
    return {
        'entities': entities,
        'relationships': relationships,
        'scope_or': scope_or
    }

In [4]:
def replace_ids_with_offsets(data):
    entities = data['entities']
    relationships = data['relationships']
    scope_or = data['scope_or']
    for relationship in relationships:
        if relationship['type'] in ["AND", "Has_negation"]:
            if relationship['arg1'] in entities and relationship['arg2'] in entities:
                start1, end1 = entities[relationship['arg1']]['start'], entities[relationship['arg1']]['end']
                start2, end2 = entities[relationship['arg2']]['start'], entities[relationship['arg2']]['end']
                # Sortierung der Offsets für konsistente Reihenfolge
                sorted_offsets = sorted([(start1, end1), (start2, end2)])
                relationship['arg1'] = f"{sorted_offsets[0][0]}-{sorted_offsets[0][1]}"
                relationship['arg2'] = f"{sorted_offsets[1][0]}-{sorted_offsets[1][1]}"
    new_scope_or = []
    for group in scope_or:
        offsets = sorted([f"{entities[id]['start']}-{entities[id]['end']}" for id in group['entities'] if id in entities], key=lambda x: int(x.split('-')[0]))
        new_scope_or.append({'type': group['type'], 'entities': offsets})
    updated_data = {
        'entities': entities, 
        'relationships': relationships,
        'scope_or': new_scope_or
    }
    return updated_data

In [5]:
ann_file = 'NCT00050349_exc.ann'
ec_file = 'NCT00050349_exc.txt'

ann_data = parse_ann_file(ann_file)
save_as_json(ann_data, 'output_file.json')
print("Conversion complete. JSON saved to 'output_file.json'")

In [6]:
def remove_last_elements(data):
    # Entferne das letzte Element aus jeder 'OR' Gruppe, wenn mehr als ein Element vorhanden ist
    for or_group in data['scope_or']:
        if len(or_group['entities']) > 1:
            or_group['entities'].pop()  # Entfernt das letzte Element

    # Entferne das letzte Argument aus den 'AND' und 'Has_negation' Beziehungen
    for relationship in data['relationships']:
        if relationship['type'] in ["AND", "Has_negation"]:
            arg1_end = int(relationship['arg1'].split('-')[1])
            arg2_end = int(relationship['arg2'].split('-')[1])
            if arg1_end > arg2_end:
                relationship['arg1'] = relationship['arg2']  # Setze arg1 auf arg2, wenn arg1 das letzte ist
            # Entferne arg2, da wir nur das erste Argument behalten
            del relationship['arg2']

    return data

In [7]:
offsets = replace_ids_with_offsets(ann_data)
offsets = remove_last_elements(offsets)

In [8]:
offsets

In [9]:
def insert_character_at_offsets(file_path, data):
    all_offsets = []
    label_for_offsets = {}
    inserted_positions = set()  # Zum Speichern bereits eingefügter Positionen

    # Verarbeiten von AND und Has_negation Beziehungen
    for relationship in data['relationships']:
        arg1_offset = f"{relationship['arg1']}"
        if relationship['type'] == "AND":
            label_for_offsets[arg1_offset] = ' [AND]'  # Nach dem Offset
            all_offsets.append(arg1_offset)
        elif relationship['type'] == "Has_negation":
            start_pos = int(arg1_offset.split('-')[0])  # Beginn des Offsets für Negation
            label_for_offsets[arg1_offset] = '[NOT] '  # Vor dem Offset
            all_offsets.append(arg1_offset)
            inserted_positions.add(start_pos)  # Markiere den Startpunkt für Negation

    # Verarbeiten von OR Gruppen
    for or_group in data['scope_or']:
        for offset in or_group['entities']:
            label_for_offsets[offset] = ' [OR]'  # Nach jedem Offset in der Gruppe
            all_offsets.append(offset)

    # Datei lesen
    with open(file_path, 'r') as file:
        content = file.read()

    # Sortieren der Offsets in umgekehrter Reihenfolge, um die Positionen korrekt zu aktualisieren
    sorted_offsets = sorted([(int(offset.split('-')[1]), offset) for offset in all_offsets], reverse=True)

    # Einfügen der Labels in den Text
    for end_pos, offset in sorted_offsets:
        if end_pos not in inserted_positions:  # Überprüfe, ob das Tag bereits eingefügt wurde
            label = label_for_offsets[offset]
            insert_pos = int(offset.split('-')[0]) if label.strip() == '[NOT]' else end_pos
            content = content[:insert_pos] + label + content[insert_pos:]
            inserted_positions.add(insert_pos)

    # Ausgabe in eine neue Datei schreiben
    with open("output_with_tags.txt", 'w') as file:
        file.write(content)

    return content


In [10]:
content = insert_character_at_offsets(ec_file, offsets)

In [11]:
content

In [12]:
def ec_to_json(text):
    sentences = text.strip().split('\n')
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    data = {f"EC{i+1}": sentence for i, sentence in enumerate(sentences)}
    json_output = json.dumps(data, indent=2, ensure_ascii=False)
    with open("parsed_1.json", "w", encoding="utf-8") as file:
        file.write(json_output)
    return json_output

In [13]:
data = ec_to_json(content)

In [15]:
def parse_to_structured_json(input_json):
    data = json.loads(input_json)

    def process_section(text, path):
        # Segmentierung des Textes basierend auf den Operatoren [OR], [AND], [NOT]
        segments = re.split(r'(\[OR\]|\[AND\]|\[NOT\])', text)
        #print(segments)  # Debug-Ausgabe der Segmente
        elements = []
        operators = []

        for segment in segments:
            segment = segment.strip()
            if segment in ['[OR]', '[AND]', '[NOT]']:
                operators.append(segment.strip('[]'))  # Sammeln der Operatoren ohne Klammern
            else:
                # Entfernung aller übriggebliebenen Tags und Streichen von Leerzeichen
                clean_segment = re.sub(r'\[\w+\]', '', segment).strip()
                if clean_segment:
                    elements.append(clean_segment)

        # Entscheidung, wie die Struktur aufgebaut sein soll, basierend auf der Anzahl der Elemente und Operatoren
        if len(elements) == 1 and not operators:
            return elements[0]  # Ein einzelnes Element ohne Operatoren
        else:
            structure = {}
            for index, element in enumerate(elements, 1):
                structure[f"{path}.{index}"] = element
            if operators:
                structure['operators'] = operators  # Speichern aller Operatoren unter einem eigenen Schlüssel
            return structure

    result_structure = {"EC": {}}
    index = 1
    for key, value in data.items():
        section_path = f"EC{index}"
        result_structure["EC"][section_path] = process_section(value, section_path)
        index += 1

    return json.dumps(result_structure, indent=4, ensure_ascii=False)

In [16]:
parsed_data = parse_to_structured_json(data)
print(parsed_data)
save_json_to_file(parsed_data, 'parsed_2.json')